# 🐪 Aero-CamelBERT-X — Phase 5 | **Ablation Sweep & Continual-Learning Benchmarking**

This notebook builds on the final Phase 4 configuration
(`aero_phase4_xai_continual_fixed_v3.ipynb`) and turns it into a
reusable, parameterized function so we can run controlled one-at-a-time
ablations instead of manually editing and re-running the notebook for
every hyperparameter value.

**What this notebook does:**
1. Rebuilds the same setup as Phase 4 (model, stream, gate, EWC — as functions)
2. Wraps the full streaming-eval loop into `run_phase4_stream(...)`, parameterized by:
   `ewc_lambda`, `adwin_delta`, `percentile_w`, `percentile_p`, `replay_n`,
   `post_drift_buffer_size`, `drift_gated`, `seed`
3. Runs a **one-at-a-time ablation sweep** around the locked-in final config
4. Computes standard continual-learning metrics — **BWT** (backward transfer /
   forgetting) and **FWT** (forward transfer / zero-shot generalization) —
   alongside the Δ_acc numbers from Phase 4
5. Runs a **seed-robustness check** on the final config (mean ± std over 5 seeds)
6. Saves one consolidated `ablation_summary.csv` + `robustness_summary.csv`
   and a comparison figure — everything Phase 5's results section needs.


## 1. Mount + reload config

In [ ]:
from google.colab import drive
import json, os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/aero_camelbert_x'
with open(f'{PROJECT_DIR}/configs/config.json') as f:
    CONFIG = json.load(f)

# Phase-5-specific outputs
P5_DIR = f'{CONFIG["log_dir"]}/phase5'
os.makedirs(P5_DIR, exist_ok=True)
os.makedirs(f'{P5_DIR}/figures', exist_ok=True)
os.makedirs(f'{P5_DIR}/runs', exist_ok=True)   # per-run CSVs land here
print('Phase-5 dir:', P5_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Phase-5 dir: /content/drive/MyDrive/aero_camelbert_x/logs/phase5


## 2. Install

Same lean install as Phase 4.

In [ ]:
!pip uninstall -y numpy scipy scikit-learn transformers river
!pip install --no-cache-dir --force-reinstall \
  numpy==2.0.2 \
  scipy==1.14.1 \
  scikit-learn==1.5.2 \
  transformers==4.44.2 \
  river

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 30.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 266.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 261.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 216.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 252.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of river to determine which version is compatible with other requirements. This c

## 3. Imports

In [ ]:
import os, json, time, copy, random, itertools
from collections import deque

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score

from river import drift as river_drift

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('numpy:', np.__version__, '| river:', __import__('river').__version__)


Device: cuda
numpy: 2.0.2 | river: 0.23.0


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


## 4. Load the corpus once

We load the raw dataframe once at the top. The drift stream itself
(the research→social shuffle+concat) is rebuilt inside `build_stream(seed)`
below, since the seed-robustness sweep needs a fresh shuffle per seed.

In [ ]:
path = CONFIG['corpus_path']
df = pd.read_csv(path) if path.endswith('.csv') else pd.read_parquet(path)
test_df  = df[df.split == 'test' ].copy().reset_index(drop=True)
train_df = df[df.split == 'train'].copy()

LABEL2ID = {'human': 0, 'ai': 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def build_stream(seed):
    research_df = test_df[test_df.domain == 'research'    ].copy()
    social_df   = test_df[test_df.domain == 'social_media'].copy()
    research_df = research_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    social_df   = social_df  .sample(frac=1, random_state=seed).reset_index(drop=True)
    stream_df = pd.concat([research_df, social_df], ignore_index=True)
    drift_at  = len(research_df)
    return stream_df, research_df, social_df, drift_at

_stream_df, _research_df, _social_df, _DRIFT_AT = build_stream(CONFIG['seed'])
print(f'Stream length: {len(_stream_df)}  |  drift at idx {_DRIFT_AT}  '
      f'(research={len(_research_df)}, social={len(_social_df)})')


Stream length: 1262  |  drift at idx 598  (research=598, social=664)


## 5. Model builder

Reloads the frozen Phase-2 checkpoint fresh for every run, so each ablation starts from identical weights.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

class AeroCamelBertX(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)

_CKPT = torch.load(f'{CONFIG["checkpoint_dir"]}/aero_baseline_best.pt',
                   map_location=device, weights_only=False)
print(f'Phase-2 baseline checkpoint loaded (val F1 {_CKPT["val_f1"]:.4f}, '
      f'epoch {_CKPT["epoch"]})')

def build_model_and_baseline():
    # Fresh continual-learning model (trainable) + frozen baseline (eval-only),
    # both starting from the identical Phase-2 checkpoint.
    model = AeroCamelBertX(CONFIG['model_name'], num_labels=CONFIG['num_labels'],
                           dropout=CONFIG['dropout']).to(device)
    model.load_state_dict(_CKPT['model_state'])
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True
    for i in [10, 11]:
        for p in model.encoder.encoder.layer[i].parameters():
            p.requires_grad = True

    baseline_model = copy.deepcopy(model).eval()
    for p in baseline_model.parameters():
        p.requires_grad = False

    return model, baseline_model


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Phase-2 baseline checkpoint loaded (val F1 0.9286, epoch 10)


## 6. Gate, EWC, and dataset classes

Unchanged from Phase 4 — reproduced here so this notebook is self-contained.

In [ ]:
class ADWINDriftDetector:
    def __init__(self, delta=0.002):
        self.adwin = river_drift.ADWIN(delta=delta)
        self.delta = delta
        self.n_seen = 0
        self.n_drifts = 0
        self.drift_indices = []

    def update(self, confidence):
        self.adwin.update(float(confidence))
        self.n_seen += 1
        if self.adwin.drift_detected:
            self.n_drifts += 1
            self.drift_indices.append(self.n_seen - 1)
            return True
        return False


class AdaptivePercentileGate:
    def __init__(self, window_size=200, percentile=20):
        self.window = deque(maxlen=window_size)
        self.p = percentile
        self.n_admitted = 0
        self.admit_indices = []
        self.n_seen = 0

    def should_admit(self, confidence):
        c = float(confidence)
        if len(self.window) < 30:
            self.window.append(c)
            self.n_seen += 1
            return False
        threshold = np.percentile(list(self.window), self.p)
        admit = c <= threshold
        self.window.append(c)
        if admit:
            self.n_admitted += 1
            self.admit_indices.append(self.n_seen)
        self.n_seen += 1
        return admit


class DualLoopGate:
    def __init__(self, adwin_delta=0.002, window_size=200, percentile=20):
        self.adwin = ADWINDriftDetector(delta=adwin_delta)
        self.pct   = AdaptivePercentileGate(window_size=window_size, percentile=percentile)
        self.events = []

    def step(self, confidence, stream_idx):
        drift = self.adwin.update(confidence)
        adm_pct = self.pct.should_admit(confidence)
        admit = drift or adm_pct
        self.events.append({
            'stream_idx'  : stream_idx,
            'confidence'  : float(confidence),
            'drift_signal': bool(drift),
            'pct_signal'  : bool(adm_pct),
            'admitted'    : bool(admit),
        })
        return admit, drift


class ArabicAIDetectionDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(row['text'], truncation=True, padding='max_length',
                             max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'label'          : torch.tensor(LABEL2ID[row['label']], dtype=torch.long),
        }


def fisher_subset_loader(df, tokenizer, seed, n_samples=512, batch_size=8):
    sub = df.sample(n=min(n_samples, len(df)), random_state=seed)
    ds  = ArabicAIDetectionDataset(sub, tokenizer, CONFIG['max_length'])
    return DataLoader(ds, batch_size=batch_size, shuffle=False)


class OnlineEWC:
    def __init__(self, model, lam=10.0):
        self.lam = lam
        self.trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
        self.anchor = {n: p.detach().clone() for n, p in model.named_parameters()
                       if n in self.trainable_names}
        self.fisher = {n: torch.zeros_like(p) for n, p in self.anchor.items()}
        self.fisher_estimated = False

    @torch.no_grad()
    def _zero_fisher(self):
        for n in self.fisher:
            self.fisher[n].zero_()

    def estimate_fisher(self, model, data_loader, max_batches=64):
        self._zero_fisher()
        model.eval()
        criterion = nn.CrossEntropyLoss()
        n_processed = 0
        for b, batch in enumerate(data_loader):
            if b >= max_batches:
                break
            model.zero_grad()
            input_ids      = batch['input_ids'     ].to(device)
            attention_mask = batch['attention_mask'].to(device)
            y              = batch['label'         ].to(device)
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, y)
            loss.backward()
            for n, p in model.named_parameters():
                if n in self.trainable_names and p.grad is not None:
                    self.fisher[n] += p.grad.detach().pow(2)
            n_processed += 1
        if n_processed > 0:
            for n in self.fisher:
                self.fisher[n] /= n_processed
        self.fisher_estimated = True

    def normalize(self):
        all_vals = torch.cat([f.flatten() for f in self.fisher.values()])
        global_mean = all_vals.mean().item()
        if global_mean > 1e-12:
            for n in self.fisher:
                self.fisher[n] /= global_mean
        return global_mean

    def penalty(self, model):
        if not self.fisher_estimated:
            return torch.tensor(0.0, device=device)
        pen = torch.tensor(0.0, device=device)
        for n, p in model.named_parameters():
            if n in self.trainable_names:
                pen = pen + (self.fisher[n] * (p - self.anchor[n]).pow(2)).sum()
        return self.lam * pen


## 7. `run_phase4_stream(...)` — the parameterized ablation runner

This is the whole Phase 4 streaming loop, wrapped as a function. Every
ablation run below is just a different set of keyword arguments to this
function — no manual notebook editing required.

Also computes two standard continual-learning metrics on top of the
Phase 4 Δ_acc numbers:

- **BWT (backward transfer / forgetting)**: re-evaluates the *final*
  continual model on the research (pre-drift) samples and compares to
  the frozen baseline's accuracy on the same samples. Negative BWT =
  forgetting occurred while adapting to the new domain.
- **FWT (forward transfer)**: accuracy on social-media samples seen
  *before* the first update fires (genuine zero-shot generalization
  from pre-drift knowledge), minus the 0.5 chance level for this
  binary task.

In [ ]:
def run_phase4_stream(run_name, ewc_lambda=10.0, adwin_delta=0.002,
                       percentile_w=200, percentile_p=20, replay_n=3,
                       pre_drift_buffer_size=3, post_drift_buffer_size=2,
                       drift_gated=True, seed=None, save_csvs=False, verbose=True):
    seed = seed if seed is not None else CONFIG['seed']
    set_seed(seed)

    stream_df, research_df, social_df, DRIFT_AT = build_stream(seed)
    model, baseline_model = build_model_and_baseline()

    gate = DualLoopGate(adwin_delta=adwin_delta, window_size=percentile_w,
                        percentile=percentile_p)

    ewc = OnlineEWC(model, lam=ewc_lambda)
    fisher_loader = fisher_subset_loader(train_df, tokenizer, seed, n_samples=512, batch_size=8)
    ewc.estimate_fisher(model, fisher_loader, max_batches=64)
    ewc.normalize()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=CONFIG['learning_rate'])
    criterion = nn.CrossEntropyLoss()

    def tokenize_one(text, max_length):
        enc = tokenizer(text, truncation=True, padding='max_length',
                        max_length=max_length, return_tensors='pt')
        return enc['input_ids'].to(device), enc['attention_mask'].to(device)

    def tokenize_batch(texts, max_length):
        enc = tokenizer(list(texts), truncation=True, padding=True,
                        max_length=max_length, return_tensors='pt').to(device)
        return enc['input_ids'], enc['attention_mask']

    results = []
    update_events = []
    buffer_texts, buffer_labels = [], []
    drift_has_fired = False
    first_update_idx = None

    model.eval()
    t_start = time.time()
    for i, row in stream_df.iterrows():
        text, y = row['text'], LABEL2ID[row['label']]

        with torch.no_grad():
            ids, mask = tokenize_one(text, CONFIG['max_length'])
            logits = model(ids, mask)
            conf_cont = float(F.softmax(logits, dim=-1).max().item())
            pred_cont = int(logits.argmax(-1).item())
            logits_b = baseline_model(ids, mask)
            pred_b = int(logits_b.argmax(-1).item())

        admit, drift_flag = gate.step(conf_cont, stream_idx=i)

        if drift_flag:
            if not drift_has_fired:
                drift_has_fired = True
            ewc.anchor = {n: p.detach().clone() for n, p in model.named_parameters()
                           if n in ewc.trainable_names}
            fresh_loader = fisher_subset_loader(train_df, tokenizer, seed,
                                                 n_samples=256, batch_size=8)
            ewc.estimate_fisher(model, fresh_loader, max_batches=8)
            ewc.normalize()

        effective_admit = admit and (drift_has_fired or not drift_gated)
        if effective_admit:
            buffer_texts.append(text)
            buffer_labels.append(y)

        active_buffer_size = post_drift_buffer_size if drift_has_fired else pre_drift_buffer_size
        if len(buffer_texts) >= active_buffer_size:
            model.train()
            for j in range(10):
                for p in model.encoder.encoder.layer[j].parameters():
                    p.requires_grad = False
                model.encoder.encoder.layer[j].eval()

            if replay_n > 0:
                replay_sample = research_df.sample(n=min(replay_n, len(research_df)))
                combined_texts  = buffer_texts + replay_sample['text'].tolist()
                combined_labels = buffer_labels + [LABEL2ID[l] for l in replay_sample['label'].tolist()]
                n_replay = len(replay_sample)
            else:
                combined_texts, combined_labels, n_replay = buffer_texts, buffer_labels, 0

            ids_b, mask_b = tokenize_batch(combined_texts, CONFIG['max_length'])
            y_b = torch.tensor(combined_labels, dtype=torch.long, device=device)

            optimizer.zero_grad()
            out = model(ids_b, mask_b)
            loss_task = criterion(out, y_b)
            loss_ewc  = ewc.penalty(model)
            loss = loss_task + loss_ewc
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()

            if first_update_idx is None:
                first_update_idx = i

            update_events.append({
                'stream_idx': i, 'loss_task': float(loss_task.item()),
                'loss_ewc': float(loss_ewc.item()), 'buffer_n': len(buffer_texts),
                'replay_n': n_replay,
            })
            buffer_texts.clear(); buffer_labels.clear()
            model.eval()

        results.append({
            'stream_idx': i, 'domain': row['domain'], 'label': y,
            'pred_cont': pred_cont, 'pred_base': pred_b, 'confidence': conf_cont,
            'correct_cont': int(pred_cont == y), 'correct_base': int(pred_b == y),
            'admitted': int(effective_admit),
        })

    results_df = pd.DataFrame(results)
    updates_df = pd.DataFrame(update_events)

    # ---- Segment metrics (Phase 4 headline numbers) ----
    def seg_metrics(sub, name):
        return {
            'segment': name, 'n': len(sub),
            'acc_base': accuracy_score(sub['label'], sub['pred_base']),
            'acc_cont': accuracy_score(sub['label'], sub['pred_cont']),
            'delta_acc': accuracy_score(sub['label'], sub['pred_cont']) - accuracy_score(sub['label'], sub['pred_base']),
        }

    pre  = results_df[results_df.stream_idx <  DRIFT_AT]
    post = results_df[results_df.stream_idx >= DRIFT_AT]
    last200 = results_df.tail(200)

    seg_pre, seg_post, seg_last, seg_all = (seg_metrics(pre, 'pre'), seg_metrics(post, 'post'),
                                             seg_metrics(last200, 'last200'),
                                             seg_metrics(results_df, 'overall'))

    # ---- BWT: re-evaluate final model on research (pre-drift) domain ----
    model.eval()
    research_preds = []
    with torch.no_grad():
        for _, r in research_df.iterrows():
            ids, mask = tokenize_one(r['text'], CONFIG['max_length'])
            research_preds.append(int(model(ids, mask).argmax(-1).item()))
    acc_research_final = accuracy_score([LABEL2ID[l] for l in research_df['label']], research_preds)
    bwt = acc_research_final - seg_pre['acc_base']   # baseline pre-drift acc == acc right after task 1

    # ---- FWT: accuracy on social samples seen before the first update fired ----
    if first_update_idx is not None:
        fwt_window = results_df[(results_df.stream_idx >= DRIFT_AT) & (results_df.stream_idx < first_update_idx)]
    else:
        fwt_window = post
    fwt = (fwt_window['correct_cont'].mean() if len(fwt_window) else float('nan')) - 0.5

    n_updates = len(updates_df)
    n_pre_updates = int((updates_df['stream_idx'] < DRIFT_AT).sum()) if n_updates else 0
    ewc_ratio = float(updates_df['loss_ewc'].mean() / updates_df['loss_task'].mean()) if n_updates else float('nan')
    drift_idx = gate.adwin.drift_indices[0] if gate.adwin.n_drifts else None

    row = {
        'run_name': run_name, 'ewc_lambda': ewc_lambda, 'adwin_delta': adwin_delta,
        'percentile_w': percentile_w, 'percentile_p': percentile_p, 'replay_n': replay_n,
        'pre_drift_buffer_size': pre_drift_buffer_size, 'post_drift_buffer_size': post_drift_buffer_size,
        'drift_gated': drift_gated, 'seed': seed,
        'pre_delta_acc': seg_pre['delta_acc'], 'post_delta_acc': seg_post['delta_acc'],
        'last200_delta_acc': seg_last['delta_acc'], 'overall_delta_acc': seg_all['delta_acc'],
        'n_updates': n_updates, 'n_pre_drift_updates': n_pre_updates,
        'ewc_task_ratio': ewc_ratio, 'drift_detected_idx': drift_idx,
        'drift_delay': (drift_idx - DRIFT_AT) if drift_idx is not None else None,
        'bwt_research': bwt, 'fwt_social': fwt,
        'elapsed_min': (time.time() - t_start) / 60,
    }

    if verbose:
        print(f"[{run_name}] overall Δacc={row['overall_delta_acc']:+.4f}  "
              f"pre={row['pre_delta_acc']:+.4f}  post={row['post_delta_acc']:+.4f}  "
              f"last200={row['last200_delta_acc']:+.4f}  BWT={bwt:+.4f}  FWT={fwt:+.4f}  "
              f"updates={n_updates} ({n_pre_updates} pre-drift)  "
              f"({row['elapsed_min']:.1f} min)")

    if save_csvs:
        run_dir = f'{P5_DIR}/runs/{run_name}'
        os.makedirs(run_dir, exist_ok=True)
        results_df.to_csv(f'{run_dir}/stream_results.csv', index=False)
        updates_df.to_csv(f'{run_dir}/update_events.csv', index=False)

    del model, baseline_model, ewc, gate
    torch.cuda.empty_cache()
    return row


## 8. Locked-in final config (from Phase 4)

This is `aero_phase4_xai_continual_fixed_v3` — the best result so far
(overall Δ_acc = −0.0055). Every ablation below varies exactly one
parameter away from this baseline.

In [ ]:
FINAL_CONFIG = dict(
    ewc_lambda=10.0, adwin_delta=CONFIG['adwin_delta'],
    percentile_w=CONFIG['percentile_w'], percentile_p=CONFIG['percentile_p'],
    replay_n=3, pre_drift_buffer_size=3, post_drift_buffer_size=2,
    drift_gated=True, seed=CONFIG['seed'],
)
print('Final config:', FINAL_CONFIG)

# Sanity check: reproduce the Phase 4 final numbers before sweeping anything
baseline_row = run_phase4_stream('final_v3_baseline', save_csvs=True, **FINAL_CONFIG)


Final config: {'ewc_lambda': 10.0, 'adwin_delta': 0.002, 'percentile_w': 200, 'percentile_p': 20, 'replay_n': 3, 'pre_drift_buffer_size': 3, 'post_drift_buffer_size': 2, 'drift_gated': True, 'seed': 42}


pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

[final_v3_baseline] overall Δacc=-0.0024  pre=+0.0000  post=-0.0045  last200=+0.0050  BWT=-0.0134  FWT=+0.3967  updates=49 (0 pre-drift)  (0.8 min)


## 9. One-at-a-time ablation sweep

Each block below holds every parameter at `FINAL_CONFIG` except the one
under test.

In [ ]:
ablation_rows = [baseline_row]

# ---- EWC lambda ----
for lam in [5, 20, 40]:
    cfg = {**FINAL_CONFIG, 'ewc_lambda': lam}
    ablation_rows.append(run_phase4_stream(f'ewc_lambda_{lam}', save_csvs=False, **cfg))

# ---- ADWIN delta ----
for delta in [0.001, 0.01, 0.05]:
    cfg = {**FINAL_CONFIG, 'adwin_delta': delta}
    ablation_rows.append(run_phase4_stream(f'adwin_delta_{delta}', save_csvs=False, **cfg))

# ---- Replay N ----
for rn in [0, 1, 5]:
    cfg = {**FINAL_CONFIG, 'replay_n': rn}
    ablation_rows.append(run_phase4_stream(f'replay_n_{rn}', save_csvs=False, **cfg))

# ---- Post-drift buffer size ----
for bs in [1, 3, 5]:
    cfg = {**FINAL_CONFIG, 'post_drift_buffer_size': bs}
    ablation_rows.append(run_phase4_stream(f'post_drift_buffer_{bs}', save_csvs=False, **cfg))

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(f'{P5_DIR}/ablation_summary.csv', index=False)
ablation_df


[ewc_lambda_5] overall Δacc=-0.0079  pre=+0.0000  post=-0.0151  last200=-0.0350  BWT=-0.0151  FWT=+0.3967  updates=45 (0 pre-drift)  (0.8 min)
[ewc_lambda_20] overall Δacc=-0.0024  pre=+0.0000  post=-0.0045  last200=+0.0050  BWT=-0.0050  FWT=+0.3967  updates=49 (0 pre-drift)  (0.8 min)
[ewc_lambda_40] overall Δacc=-0.0040  pre=+0.0000  post=-0.0075  last200=+0.0100  BWT=-0.0050  FWT=+0.3967  updates=47 (0 pre-drift)  (0.8 min)
[adwin_delta_0.001] overall Δacc=-0.0127  pre=+0.0000  post=-0.0241  last200=-0.0400  BWT=-0.0184  FWT=+0.3951  updates=39 (0 pre-drift)  (0.8 min)
[adwin_delta_0.01] overall Δacc=-0.0055  pre=+0.0000  post=-0.0105  last200=-0.0250  BWT=-0.0033  FWT=+0.3844  updates=53 (0 pre-drift)  (0.8 min)
[adwin_delta_0.05] overall Δacc=-0.0127  pre=+0.0000  post=-0.0241  last200=-0.0250  BWT=-0.0151  FWT=+0.4018  updates=54 (0 pre-drift)  (0.8 min)
[replay_n_0] overall Δacc=-0.0024  pre=+0.0000  post=-0.0045  last200=+0.0050  BWT=-0.0033  FWT=+0.3967  updates=43 (0 pre-drif

,run_name,ewc_lambda,adwin_delta,percentile_w,percentile_p,replay_n,pre_drift_buffer_size,post_drift_buffer_size,drift_gated,seed,...,last200_delta_acc,overall_delta_acc,n_updates,n_pre_drift_updates,ewc_task_ratio,drift_detected_idx,drift_delay,bwt_research,fwt_social,elapsed_min
0,final_v3_baseline,10.0,0.002,200,20,3,3,2,True,42,...,0.005,-0.002377,49,0,0.256020,831,233,-0.013378,0.396694,0.806781
1,ewc_lambda_5,5.0,0.002,200,20,3,3,2,True,42,...,-0.035,-0.007924,45,0,0.156397,831,233,-0.015050,0.396694,0.800091
2,ewc_lambda_20,20.0,0.002,200,20,3,3,2,True,42,...,0.005,-0.002377,49,0,0.368101,831,233,-0.005017,0.396694,0.810502
3,ewc_lambda_40,40.0,0.002,200,20,3,3,2,True,42,...,0.010,-0.003962,47,0,0.513266,831,233,-0.005017,0.396694,0.811633
4,adwin_delta_0.001,10.0,0.001,200,20,3,3,2,True,42,...,-0.040,-0.012678,39,0,0.230785,863,265,-0.018395,0.395131,0.800712
5,adwin_delta_0.01,10.0,0.010,200,20,3,3,2,True,42,...,-0.025,-0.005547,53,0,0.286710,767,169,-0.003344,0.384393,0.823845
6,adwin_delta_0.05,10.0,0.050,200,20,3,3,2,True,42,...,-0.025,-0.012678,54,0,0.238111,703,105,-0.015050,0.401786,0.831548
7,replay_n_0,10.0,0.002,200,20,0,3,2,True,42,...,0.005,-0.002377,43,0,0.150096,831,233,-0.003344,0.396694,0.781750
8,replay_n_1,10.0,0.002,200,20,1,3,2,True,42,...,-0.035,-0.009509,44,0,0.197435,831,233,-0.003344,0.396694,0.782460
9,replay_n_5,10.0,0.002,200,20,5,3,2,True,42,...,-0.020,-0.006339,48,0,0.269635,831,233,0.000000,0.396694,0.838408


## 10. Seed-robustness check on the final config

Replay sampling is stochastic, so a single run isn't fully representative.
This reruns `FINAL_CONFIG` across 5 seeds and reports mean ± std.

In [ ]:
SEEDS = [42, 1, 7, 123, 2024]
robustness_rows = []
for s in SEEDS:
    cfg = {**FINAL_CONFIG, 'seed': s}
    robustness_rows.append(run_phase4_stream(f'final_v3_seed{s}', save_csvs=False, **cfg))

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(f'{P5_DIR}/robustness_summary.csv', index=False)

summary_cols = ['pre_delta_acc', 'post_delta_acc', 'last200_delta_acc',
                'overall_delta_acc', 'bwt_research', 'fwt_social']
print('=== Final config — mean ± std over 5 seeds ===')
for c in summary_cols:
    print(f'  {c:<20}: {robustness_df[c].mean():+.4f} ± {robustness_df[c].std():.4f}')

robustness_df


[final_v3_seed42] overall Δacc=-0.0024  pre=+0.0000  post=-0.0045  last200=+0.0050  BWT=-0.0134  FWT=+0.3967  updates=49 (0 pre-drift)  (0.8 min)
[final_v3_seed1] overall Δacc=-0.0040  pre=+0.0000  post=-0.0075  last200=-0.0200  BWT=+0.0000  FWT=+0.4106  updates=43 (0 pre-drift)  (0.8 min)
[final_v3_seed7] overall Δacc=-0.0040  pre=+0.0000  post=-0.0075  last200=-0.0150  BWT=-0.0017  FWT=+0.3835  updates=45 (0 pre-drift)  (0.8 min)
[final_v3_seed123] overall Δacc=+0.0000  pre=+0.0000  post=+0.0000  last200=+0.0100  BWT=-0.0033  FWT=+0.3865  updates=55 (0 pre-drift)  (0.8 min)
[final_v3_seed2024] overall Δacc=-0.0071  pre=+0.0000  post=-0.0136  last200=-0.0050  BWT=-0.0017  FWT=+0.4043  updates=46 (0 pre-drift)  (0.8 min)
=== Final config — mean ± std over 5 seeds ===
  pre_delta_acc       : +0.0000 ± 0.0000
  post_delta_acc      : -0.0066 ± 0.0049
  last200_delta_acc   : -0.0050 ± 0.0127
  overall_delta_acc   : -0.0035 ± 0.0026
  bwt_research        : -0.0040 ± 0.0054
  fwt_social     

,run_name,ewc_lambda,adwin_delta,percentile_w,percentile_p,replay_n,pre_drift_buffer_size,post_drift_buffer_size,drift_gated,seed,...,last200_delta_acc,overall_delta_acc,n_updates,n_pre_drift_updates,ewc_task_ratio,drift_detected_idx,drift_delay,bwt_research,fwt_social,elapsed_min
0,final_v3_seed42,10.0,0.002,200,20,3,3,2,True,42,...,0.005,-0.002377,49,0,0.256020,831,233,-0.013378,0.396694,0.803998
1,final_v3_seed1,10.0,0.002,200,20,3,3,2,True,1,...,-0.020,-0.003962,43,0,0.252039,831,233,0.000000,0.410638,0.800097
2,final_v3_seed7,10.0,0.002,200,20,3,3,2,True,7,...,-0.015,-0.003962,45,0,0.303082,799,201,-0.001672,0.383495,0.806456
3,final_v3_seed123,10.0,0.002,200,20,3,3,2,True,123,...,0.010,0.000000,55,0,0.230319,735,137,-0.003344,0.386525,0.826290
4,final_v3_seed2024,10.0,0.002,200,20,3,3,2,True,2024,...,-0.005,-0.007132,46,0,0.213853,799,201,-0.001672,0.404306,0.813144


## 11. Ablation comparison figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=True)
groups = [
    ('ewc_lambda',            [5, 10, 20, 40],              axes[0,0], 'EWC lambda'),
    ('adwin_delta',           [0.001, 0.002, 0.01, 0.05],   axes[0,1], 'ADWIN delta'),
    ('replay_n',              [0, 1, 3, 5],                 axes[1,0], 'Replay N'),
    ('post_drift_buffer_size',[1, 2, 3, 5],                 axes[1,1], 'Post-drift buffer size'),
]

for param, values, ax, label in groups:
    ys = []
    for v in values:
        match = ablation_df[(ablation_df[param] == v)]
        # fall back to the baseline row when v equals the final-config value
        if match.empty and v == FINAL_CONFIG[param]:
            match = ablation_df[ablation_df['run_name'] == 'final_v3_baseline']
        ys.append(match['overall_delta_acc'].iloc[0] if not match.empty else np.nan)
    ax.plot([str(v) for v in values], ys, marker='o', color='#1f4e79')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(label)
    ax.set_ylabel('Overall Δ_acc')
    ax.grid(alpha=0.3)

plt.suptitle('Phase 5 ablation sweep — overall Δ_acc vs. final config', fontsize=13)
plt.tight_layout()
plt.savefig(f'{P5_DIR}/figures/ablation_sweep.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {P5_DIR}/figures/ablation_sweep.png')


Saved: /content/drive/MyDrive/aero_camelbert_x/logs/phase5/figures/ablation_sweep.png


## 12. Wrap-up

What you have now:

| File | What |
|---|---|
| `ablation_summary.csv` | One row per hyperparameter variant, all Δ_acc/BWT/FWT metrics |
| `robustness_summary.csv` | Final config across 5 seeds — mean ± std |
| `figures/ablation_sweep.png` | 4-panel comparison chart |
| `runs/final_v3_baseline/` | Full stream_results/update_events for the reproduced final config |

**Next step:** once you've reviewed which direction (if any) beats
`FINAL_CONFIG`, either lock in a new final config and note the ablation
findings in the Phase 5 discussion, or — if `FINAL_CONFIG` remains best
on every axis — that's a legitimate and useful negative result showing
your Phase 4 tuning already found a local optimum.


In [ ]:
print('=== Phase 5 ablation sweep complete ===')
print(f'Ablation rows : {len(ablation_df)}')
print(f'Seeds tested  : {len(robustness_df)}')
print(f'Best overall Δ_acc in sweep: '
      f'{ablation_df.loc[ablation_df.overall_delta_acc.idxmax(), "run_name"]} '
      f'({ablation_df.overall_delta_acc.max():+.4f})')


=== Phase 5 ablation sweep complete ===
Ablation rows : 13
Seeds tested  : 5
Best overall Δ_acc in sweep: post_drift_buffer_5 (-0.0016)
